In [ ]:
# Transformer 모델 구축 - Transformer RAG(Retriever Augmentation Generation) 구성 및 모델 파이프라인 구축(Qdrant 의미 기반 검색엔진 적용)
# - Retrieval 단계: Qdrant 의미 기반 검색엔진을 통해 관련 문서를 빠르게 찾아냄
# - Augmentation 단계: 검색된 문서를 기반으로 LLM 입력 프롬프트를 강화 → 맥락 있는 답변 생성
# - Generation 단계: LLM이 최종 답변을 생성 → 단순 생성보다 정확성과 신뢰성이 높아짐
# - 예시) 구글, 네이버에서 사용되는 RAG 시스템 구현

# 학습 목표 - 실무에서 사용되는 파이프라인 이해 및 적용
# 1. 데이터 수집기
# - 수집 대상 도메인: Google News RSS
# - RSS 피드 파싱: feedparser 라이브러리로 RSS XML을 읽고 기사 항목 추출
# - 데이터 정제: title,link,description,pubDate,soure 필드 추출, pubDate -> Python datetime 변화
# - DB 저장: PostgreSQL 테이블 news_articles 에 Insert
# - 수집 데이터 -> PostgreSQL 테이블 생성 및 입력
# - DB 조회 + 로깅 설정으로 데이터 관리

# 2. Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

# 3. 임베딩 생성 후 Qdrant Collection에 Insert/Update
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 4. Qdrant 의미 기반 검색
# - 의미 기반 검색으로 관련 문서 조회

# 5. QA 처리
# - Qdrant 검색 결과 -> QA 토크나이저 + QA 모델
# - 형태소 분석으로 한국어 처리 강화
# - 문자열 -> 토큰 ID 변환, 예시: "AI는 의료 분야에서 활용된다." -> [101, 1234, 5678, ...]
# - 모델 입력: input_ids 토큰 ID 배열, attention_mask 패딩 여부 표시, 모델 출력을 텍스트로 복원 [101, 1234, 5678, ...] -> "AI는 의료 분야에서 활용된다."

# 6. 요약 처리
# - 요약 토크나이저 + 요약 모델 (KoBART 기반)
# - 반복 억제, 길이 조절, 다양성 확보 등 파라미터 튜닝
# - 후처리(clean_summary)로 중복 제거
# - from transformers import BartTokenizer # 영어 전용이라 맞지 않음

# 7. 최종 응답: 외부 서비스 연계 검토
# - Local LLM은 GPU 장비 한계로 로컬 실행은 패스 -> 대신 Copilot에 문의하여 자연스러운 답변 재구성
# - 검토: 현재 로컬 GPU 장비로는 생성형 LLM 모델을 도메인에 맞추어 파인튜닝은 불가능, 현재 추론 모델도 좋은 성능의 모델로 교체 불가능

# 8. 서비스: 구글, 네이버 RAG 시스템 구성
# - /llm_app/transformer_rag2_23_app.py
# - FastAPI 구동 정보: 터미널에서 구동, uvicorn transformer_rag2_23_app:app --reload, 경로 포함 uvicorn LLM.llm_app.transformer_rag2_23_app:app --reload
# - FastAPI 서비스: /search, 입력: 질의문, 출력: QA + 요약 결과 + 출처 정보
# - 1)  [사용자 질의]
# - 2)  [FastAPI 엔드포인트: /search]
# - 3)  [SentenceTransformer: 임베딩]
# - 4)  [Qdrant 의미 기반 검색]
# - 5)  [검색 결과 문서]
# - 6)  [KoELECTRA QA 모델 + MeCab 후처리(형태소 분석: 한국어 처리 강황)]
# - 7)  [KoBART Summarization 모델 + clean_summary]
# - 8)  [응답 + 출처 표시]
# - 9)  [최종 응답 LLM 모델은  외부 서비스 연계 검토: 자연스러운 문장]
# - 10) [최종 사용자 응답]

In [11]:
# 데이터 수집기: Google News RSS
import feedparser
from bs4 import BeautifulSoup
from datetime import datetime
import psycopg2
import logging

# 로깅 설정: 파일 저장
logging.basicConfig(
    level=logging.INFO, # 로그 레벨: INFO 이상 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 로그 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 로그를 파일에 저장
        logging.StreamHandler() # 콘솔에도 출력
    ]
)

# DB 연결 한번만 수행
conn = psycopg2.connect(
    dbname="newsdb",      # 생성한 DB 이름
    user="newsuser",      # 생성한 사용자
    password="1234",      # 설정한 비밀번호
    host="localhost",     # 로컬에서 실행 중
    port="5432"           # 기본 포트
)
cur = conn.cursor()

# Google News RSS -> DB Insert
def insert_article(title, content, url, published_at, source_name, source_url):
    try: # 중복 뉴스는 무시되고, 새로운 뉴스만 저장: ON CONFLICT (url) url 컬럼 값이 이미 존재하면 충동 발생, DO NOTHING 충돌시 아무것도 하지 않고 넘어감
        cur.execute("""
            INSERT INTO news_articles (title, content, url, published_at, source_name, source_url)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING;
        """, (title, content, url, published_at, source_name, source_url))
        if cur.rowcount > 0: # 새로운 행 추가시 1
            logging.info("데이터 Insert 성공: %s", title)
        else: # 새로운 행 추가 없을시 0
            logging.info("중복으로 건너뜀: %s", title)
        conn.commit()
    except Exception as e:
        logging.error("Insert 실패: %s, 에러: %s", title, e)

# Google News RSS
rss_url = "https://news.google.com/rss?hl=ko&gl=KR&ceid=KR:ko"
feed = feedparser.parse(rss_url)

for entry in feed.entries:
    title = entry.title # title
    
    # description HTML 제거
    content_raw = entry.description if "description" in entry else ""
    content = BeautifulSoup(content_raw, "html.parser").get_text()

    url = entry.link # url
    # pubDate 처리
    if hasattr(entry, "published_parsed"): # hasattr() 객체에 안전한 속성 접근을 위해 사용
        published_at = datetime(*entry.published_parsed[:6]) # feedparser가 RSS <pubDate> 태그를 자동으로 읽어와서 속성으로 제공
    else:
        published_at = datetime.now()
    # source 처리
    if hasattr(entry, "source"):
        source_name = entry.source.title # 언론사 이름(본문 텍스트)
        source_url = entry.source.href # 언론사 URL(속성)
    else:
        source_name = "Google News"
        source_url = ""

    # DB Insert 함수
    insert_article(title, content, url, published_at, source_name, source_url)
    # print(f"{title}\n{content}\n{url}\n{published_at}\n{source_name}\n{source_url}") # 데이터 확인

# 마지막 연결 객체 닫기: 순서 지켜서 닫아야 함
cur.close()
conn.close()
logging.info("DB 연결 종료 완료")

2026-03-19 14:45:08,073 [INFO] 중복으로 건너뜀: 트럼프 “이란, 무고한 카타르 또 공격하면 가스전 날려버릴 것” 경고 - 조선일보
2026-03-19 14:45:08,074 [INFO] 중복으로 건너뜀: 남양주 스토킹 살인 피의자 신상 공개…44살 김훈 - 한겨레
2026-03-19 14:45:08,076 [INFO] 중복으로 건너뜀: 정청래 ‘김어준 출연’에 강득구는 ‘보이콧 선언’…친청계는 단골, 친명계는 발길 뜸해 - 경향신문
2026-03-19 14:45:08,077 [INFO] 중복으로 건너뜀: 트럼프의 이란 ‘하르그섬 점령’ 계획, 38년 전 머릿속에 있었다 - 조선일보
2026-03-19 14:45:08,078 [INFO] 중복으로 건너뜀: 역대급 BTS 광화문 공연, ‘라이브쇼 거장’ 수퍼볼 연출가가 진두지휘 - 조선일보
2026-03-19 14:45:08,080 [INFO] 중복으로 건너뜀: ‘호르무즈 파병’ 압박한 美, 자국 기뢰제거함은 전선 6000㎞ 밖 정박 - 조선일보
2026-03-19 14:45:08,082 [INFO] 중복으로 건너뜀: 국힘 김영환 ‘항의 삭발’…“날 컷오프 할 수 있는 건 충북도민뿐” - 한겨레
2026-03-19 14:45:08,083 [INFO] 중복으로 건너뜀: 주호영 “이진숙 대구 전략공천, 대구 시민 무시하는 것” - 동아일보
2026-03-19 14:45:08,084 [INFO] 중복으로 건너뜀: 장동혁에 ‘2차전’ 벼르는 오세훈 “혁신선대위 반드시 관철” - 한겨레
2026-03-19 14:45:08,086 [INFO] 중복으로 건너뜀: 3차례 가정방문 담임 ‘신고’ 있었지만···울산 30대 아빠와 어린 네 자녀 사망 못 막았다 - 경향신문
2026-03-19 14:45:08,088 [INFO] 중복으로 건너뜀: 미국, 이란 전쟁에 수천 명 추가파병 검토 - MBC 뉴스
2026-03-19 14:45:08,089 [INFO] 중복으로 건너뜀: 슬쩍 건너뛴 ‘이란의 핵 위협’…미